<a href="https://colab.research.google.com/github/volsarino/-/blob/main/%E3%83%8F%E3%83%B3%E3%83%89%E3%83%88%E3%83%A9%E3%83%83%E3%82%AD%E3%83%B3%E3%82%B0%E3%83%A2%E3%83%87%E3%83%AB(YOLO%E4%BD%BF%E7%94%A8).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.5 MB/s eta 0:00:00


In [ ]:
import os
import glob
import xml.etree.ElementTree as ET
import yaml
from ultralytics import YOLO
from PIL import Image, ImageDraw, ImageFont

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def convert_xml_to_yolo(xml_dir):
    xml_paths = glob.glob(os.path.join(xml_dir, '*.xml'))
    for xml_path in xml_paths:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        size = root.find('size')
        orig_w = float(size.find('width').text) if size is not None else 1.0
        orig_h = float(size.find('height').text) if size is not None else 1.0

        txt_path = xml_path.replace('.xml', '.txt')
        with open(txt_path, 'w') as f:
            for obj in root.findall('object'):
                # クラス名に関わらず手(Hand)を0として扱う場合
                cls_id = 0

                bndbox = obj.find('bndbox')
                xmin = float(bndbox.find('xmin').text)
                ymin = float(bndbox.find('ymin').text)
                xmax = float(bndbox.find('xmax').text)
                ymax = float(bndbox.find('ymax').text)

                # 中心座標・幅・高さを相対座標(0.0~1.0)に変換
                cx = ((xmin + xmax) / 2.0) / orig_w
                cy = ((ymin + ymax) / 2.0) / orig_h
                bw = (xmax - xmin) / orig_w
                bh = (ymax - ymin) / orig_h

                f.write(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")

In [ ]:
train_dir = '/content/drive/MyDrive/kaggle用/画像識別モデル開発/Dataset/train'
valid_dir = '/content/drive/MyDrive/kaggle用/画像識別モデル開発/Dataset/valid'
test_dir  = '/content/drive/MyDrive/kaggle用/画像識別モデル開発/Dataset/test'

In [ ]:
convert_xml_to_yolo(train_dir)
convert_xml_to_yolo(valid_dir)
convert_xml_to_yolo(test_dir)

In [ ]:
yaml_content = {
    'path': '/content/drive/MyDrive/kaggle用/画像識別モデル開発/Dataset',
    'train': 'train',
    'val': 'valid',
    'test': 'test',
    'names': {
        0: 'hand'
    }
}
yaml_path = '/content/drive/MyDrive/kaggle用/画像識別モデル開発/Dataset/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f)

In [ ]:
model = YOLO('yolov8n.pt')
results = model.train(
    data='/content/drive/MyDrive/kaggle用/画像識別モデル開発/Dataset/data.yaml',
    epochs=20,          # エポック数
    imgsz=640,          # YOLOの推奨画像サイズ
    batch=16,
    device=0,           # GPUを使用
    project='/content/drive/MyDrive/kaggle用/画像識別モデル開発/runs',
    name='hand_yolo_model'
)

Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/kaggle用/画像識別モデル開発/Dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ha

In [ ]:
best_model_path = '/content/drive/MyDrive/kaggle用/画像識別モデル開発/runs/hand_yolo_model-2/weights/best.pt'
model = YOLO(best_model_path)

# 動画ファイルの指定
input_video = '/content/drive/MyDrive/kaggle用/画像識別モデル開発/sample01.mp4'

# 推論実行（自動で枠を描画して保存されます）
model.predict(
    source=input_video,
    save=True,          # 結果動画を保存
    conf=0.5,           # 確信度の閾値
    device=0
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/489) /content/drive/MyDrive/kaggle用/画像識別モデル開発/sample01.mp4: 384x640 1 hand, 6.5ms
video 1/1 (frame 2/489) /content/drive/MyDrive/kaggle用/画像識別モデル開発/sample01.mp4: 384x640 1 hand, 7.9ms
video 1/1 (frame 3/489) /content/drive/MyDrive/kaggle用/画像識別モデル開発/sample01.mp4: 384x640 1 hand, 6.6ms
video 1/1 (frame 4/489) /content/drive/MyDrive/kaggle用/画像識別モデル開発/sample01.mp4: 384x640 1 hand, 6.7ms
video 1/1 (frame 5/489) /content/drive/MyDrive/kaggle用

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 depth: None
 keypoints: None
 masks: None
 names: {0: 'hand'}
 obb: None
 orig_img: array([[[ 11,  48, 122],
         [ 14,  51, 125],
         [ 16,  53, 127],
         ...,
         [ 58,  99, 144],
         [ 58,  99, 144],
         [ 58,  99, 144]],
 
        [[ 15,  52, 126],
         [ 16,  53, 127],
         [ 17,  54, 128],
         ...,
         [ 59, 100, 145],
         [ 59, 100, 145],
         [ 58,  99, 144]],
 
        [[ 16,  55, 122],
         [ 15,  54, 121],
         [ 15,  53, 123],
         ...,
         [ 60, 101, 146],
         [ 60, 101, 146],
         [ 61, 102, 147]],
 
        ...,
 
        [[ 14,  15,  20],
         [ 14,  15,  20],
         [ 14,  15,  20],
         ...,
         [ 85, 137, 186],
         [ 84, 136, 185],
         [ 86, 138, 187]],
 
        [[ 14,  15,  20],
         [ 14,  15,  20],
         [ 14,  15,  20],
         ...,
       

In [ ]:
output_project = '/content/drive/MyDrive/kaggle用/画像識別モデル開発/runs'
output_name = 'predict_results'

results = model.predict(
    source=input_video,
    save=True,
    conf=0.5,
    device=0,
    project=output_project,  # 保存先ルートを Google Drive に指定
    name=output_name,         # 保存用フォルダ名
    exist_ok=True            # 上書き保存を許可
)

print(f"動画の保存先: {output_project}/{output_name}/sample01.mp4")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/489) /content/drive/MyDrive/kaggle用/画像識別モデル開発/sample01.mp4: 384x640 1 hand, 6.6ms
video 1/1 (frame 2/489) /content/drive/MyDrive/kaggle用/画像識別モデル開発/sample01.mp4: 384x640 1 hand, 6.6ms
video 1/1 (frame 3/489) /content/drive/MyDrive/kaggle用/画像識別モデル開発/sample01.mp4: 384x640 1 hand, 7.2ms
video 1/1 (frame 4/489) /content/drive/MyDrive/kaggle用/画像識別モデル開発/sample01.mp4: 384x640 1 hand, 8.2ms
video 1/1 (frame 5/489) /content/drive/MyDrive/kaggle用